# Chapter 5 — MIMII CNN baseline (log-Mel spectrogram)

- **New notebook** (separate from the MFCC + SVM/RF baseline).
- Downloads **`-6_dB_fan.zip`** from Zenodo as `fan1.zip`, then extracts **only** `fan/id_00/*` into `fan1/` (no full unzip).
- **Binary classification:** normal = 0, abnormal = 1.
- **Input:** log-Mel spectrogram → lightweight **CNN** (PyTorch).
- **Split:** 70 / 15 / 15 stratified; metrics on test + **`cnn_cm.png`**.

In [ ]:
# --- Configuration ---
ZIP_URL = "https://zenodo.org/records/3384388/files/-6_dB_fan.zip?download=1"
ZIP_NAME = "fan1.zip"

MACHINE_TYPE = "fan"
MACHINE_ID = "id_00"
# If zip entries are like "some_root/fan/id_00/...", set e.g. "some_root/"
ZIP_INNER_PREFIX = ""

EXTRACT_ROOT_NAME = "fan1"
SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 4.0

N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 64
TARGET_FRAMES = 128  # fixed time dimension for CNN

BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3
RANDOM_STATE = 42
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15

CM_PATH = "cnn_cm.png"

In [ ]:
import os
import shutil
import warnings
import zipfile
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)

WORKING = Path("/kaggle/working")
ZIP_PATH = WORKING / ZIP_NAME
EXTRACT_ROOT = WORKING / EXTRACT_ROOT_NAME

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# 1) Download fan archive (creates fan1.zip)
if ZIP_PATH.exists():
    print("Already present:", ZIP_PATH)
else:
    !wget -q --show-progress -O fan1.zip "https://zenodo.org/records/3384388/files/-6_dB_fan.zip?download=1"
    print("Downloaded:", ZIP_PATH)

In [ ]:
def normalize_zip_name(name: str) -> str:
    return name.replace("\\", "/").lstrip("./")


def selective_extract(zip_path: Path, prefix: str, dest_dir: Path) -> None:
    """Extract only members whose path starts with prefix (no full unzip)."""
    prefix = prefix.strip("/") + "/"
    if dest_dir.exists():
        shutil.rmtree(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    with zipfile.ZipFile(zip_path, "r") as zf:
        for m in zf.namelist():
            norm = normalize_zip_name(m)
            if norm.endswith("/"):
                continue
            if not norm.startswith(prefix):
                continue
            zf.extract(m, path=dest_dir)
            n += 1
    print(f"Extracted {n} files with prefix {prefix!r} -> {dest_dir}")


# 2) Extract ONLY fan/id_00/* into fan1/
_parts = [p for p in ZIP_INNER_PREFIX.replace("\\", "/").split("/") if p]
inner_prefix = "/".join(_parts + [MACHINE_TYPE, MACHINE_ID]) + "/"
selective_extract(ZIP_PATH, inner_prefix, EXTRACT_ROOT)

audio_root = EXTRACT_ROOT.joinpath(*_parts, MACHINE_TYPE, MACHINE_ID)
normal_dir = audio_root / "normal"
abnormal_dir = audio_root / "abnormal"
for d in (normal_dir, abnormal_dir):
    if not d.is_dir():
        raise FileNotFoundError(
            f"Missing: {d}. Check ZIP_INNER_PREFIX or zip layout (list a few zf.namelist())."
        )
print("Normal wavs:", len(list(normal_dir.glob("*.wav"))))
print("Abnormal wavs:", len(list(abnormal_dir.glob("*.wav"))))

In [ ]:
def iter_segment_specs(normal_dir: Path, abnormal_dir: Path):
    """Yield (wav_path, start_sample, label) for each segment."""
    seg_len = int(SEGMENT_SECONDS * SAMPLE_RATE)
    for label, folder in ((0, normal_dir), (1, abnormal_dir)):
        for wav_path in sorted(folder.glob("*.wav")):
            y, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
            for start in range(0, len(y), seg_len):
                chunk = y[start : start + seg_len]
                if chunk.size < seg_len:
                    if chunk.size < seg_len // 2:
                        continue
                    chunk = np.pad(chunk, (0, seg_len - chunk.size))
                yield wav_path, start, label


def waveform_to_logmel(chunk: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=chunk,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    # fixed time bins
    t = log_mel.shape[1]
    if t < TARGET_FRAMES:
        log_mel = np.pad(log_mel, ((0, 0), (0, TARGET_FRAMES - t)), mode="constant")
    elif t > TARGET_FRAMES:
        log_mel = log_mel[:, :TARGET_FRAMES]
    # per-sample standardisation
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)
    return log_mel.astype(np.float32)


specs = list(iter_segment_specs(normal_dir, abnormal_dir))
paths = [s[0] for s in specs]
starts = np.array([s[1] for s in specs])
y_all = np.array([s[2] for s in specs], dtype=np.int64)
print("Segments:", len(y_all), "positive rate:", round(y_all.mean(), 3))

In [ ]:
idx = np.arange(len(y_all))
idx_train, idx_temp, y_train, y_temp = train_test_split(
    idx,
    y_all,
    test_size=(1.0 - TRAIN_FRAC),
    random_state=RANDOM_STATE,
    stratify=y_all,
)
val_ratio = VAL_FRAC / (VAL_FRAC + (1.0 - TRAIN_FRAC - VAL_FRAC))
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp,
    y_temp,
    test_size=(1.0 - val_ratio),
    random_state=RANDOM_STATE,
    stratify=y_temp,
)
print("Train/val/test counts:", len(idx_train), len(idx_val), len(idx_test))

In [ ]:
class MelSegDataset(Dataset):
    def __init__(self, indices: np.ndarray):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        wav_path, start, label = specs[j]
        seg_len = int(SEGMENT_SECONDS * SAMPLE_RATE)
        y, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True, offset=start / SAMPLE_RATE, duration=SEGMENT_SECONDS)
        if y.size < seg_len:
            y = np.pad(y, (0, seg_len - y.size))
        else:
            y = y[:seg_len]
        mel = waveform_to_logmel(y)
        x = torch.from_numpy(mel).unsqueeze(0)  # (1, n_mels, time)
        return x, int(label)


train_loader = DataLoader(
    MelSegDataset(idx_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(MelSegDataset(idx_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(MelSegDataset(idx_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_mels: int, n_frames: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((2, 4)),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128 * 2 * 4, 64), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(64, 2))

    def forward(self, x):
        return self.head(self.net(x))


model = SmallCNN(N_MELS, TARGET_FRAMES).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
counts = np.bincount(y_train, minlength=2)
w = len(y_train) / (2 * np.maximum(counts, 1))
class_weights = torch.tensor(w, dtype=torch.float32, device=DEVICE)
crit = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
def run_epoch(loader, train_mode: bool):
    if train_mode:
        model.train()
    else:
        model.eval()
    tot_loss = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        if train_mode:
            opt.zero_grad()
        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = crit(logits, yb)
        if train_mode:
            loss.backward()
            opt.step()
        tot_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return tot_loss / max(n, 1)


best_state = None
best_val = float("inf")
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    print(f"Epoch {ep:02d}  train_loss={tr:.4f}  val_loss={va:.4f}")
    if va < best_val:
        best_val = va
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)
model.to(DEVICE)

In [ ]:
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(DEVICE))
        pred = logits.argmax(dim=1).cpu().numpy()
        all_pred.append(pred)
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_pred)
y_true = np.concatenate(all_true)

print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4))
print("Recall   :", round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4))
print("F1       :", round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4))
cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix [ [TN FP] [FN TP] ]:")
print(cm)
print(classification_report(y_true, y_pred, target_names=["normal", "abnormal"]))

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["pred normal", "pred abnormal"],
    yticklabels=["true normal", "true abnormal"],
    ylabel="True label",
    xlabel="Predicted label",
    title="CNN (Mel) — confusion matrix (test)",
)
thresh = cm.max() / 2.0 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            format(cm[i, j], "d"),
            ha="center",
            va="center",
            color="white" if cm[i, j] > thresh else "black",
        )
fig.tight_layout()
out = WORKING / CM_PATH
fig.savefig(out, dpi=150)
plt.show()
print("Saved:", out)

## Notes

- If extraction fails, inspect zip layout: `zipfile.ZipFile(ZIP_PATH).namelist()[:20]` and adjust **`ZIP_INNER_PREFIX`**.
- **Internet** must be enabled on Kaggle for `wget`.
- This baseline is intentionally small; tune `EPOCHS`, `TARGET_FRAMES`, and CNN width for your thesis tables.